# Genie BEFORE: Base Silver Tables

**Run on serverless compute.**

This notebook runs five questions against the base Silver tables — the catalog as it exists before any graph enrichment. The first question is the anchor: which merchants do high-volume accounts visit most? That is the best the base catalog can do. The same question, asked against the enriched Gold tables in Genie, returns a structurally different answer. The gap between the two is the demo's argument.

The warm-up and analytics challenge confirm Genie is working and handling joins and conditional aggregates correctly. The five anchor questions show where volume-based proxies fall short of structural answers.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"

SPACE_ID = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id_before")

In [ ]:
from databricks.sdk import WorkspaceClient
from demo_utils import genie_caller

w = WorkspaceClient()
print("Connected to:", w.config.host)
ask_genie = genie_caller(w, SPACE_ID)

## Warm-Up: Tabular Baseline

Rank accounts by total spend. Confirms Genie is working and establishes the baseline before the anchor questions run.

In [ ]:
WARMUP_QUESTION = "What are the top 10 accounts by total amount spent across all merchants?"

print("[W] warm_up — asking...")
response_warmup = ask_genie(WARMUP_QUESTION)

if response_warmup["df"] is not None:
    print(f"    Rows: {len(response_warmup['df'])}")
    print(f"    SQL:  {response_warmup['sql']}")
    display(response_warmup["df"])
else:
    print(f"    Status: {response_warmup['status']}")
    print(f"    Text:   {response_warmup['text']}")

## Analytics Challenge

Accounts with both elevated total spend and above-average night activity. Night transactions (hours 0–5) are an established fraud signal. Genie resolves this with a join and a conditional aggregate — all three dimensions exist in the base tables.

**Expected outcome:** a clean top-15 list with total spend, night ratio, and balance columns.

In [ ]:
ANALYTICS_QUESTION = (
    "Which accounts have both above-average total spend and a night transaction ratio "
    "above 20%? Show the top 15 by total spend with their night ratio and account balance."
)

print("[A] analytics_challenge — asking...")
response_analytics = ask_genie(ANALYTICS_QUESTION)

if response_analytics["df"] is not None:
    print(f"    Rows: {len(response_analytics['df'])}")
    print(f"    SQL:  {response_analytics['sql']}")
    display(response_analytics["df"])
else:
    print(f"    Status: {response_analytics['status']}")
    print(f"    Text:   {response_analytics['text']}")

## Anchor Questions

The next five questions are the anchor set. Each uses a volume or frequency proxy as a stand-in for the structural dimension it cannot reach. These are the best answers the base catalog can produce. The same questions, asked against the enriched catalog in Genie, return structurally different results because `community_id`, `fraud_risk_tier`, and `risk_score` now exist as columns.

---

### 1. Merchant Favorites

Which merchants do the highest-volume accounts visit most? On the base catalog, volume is the only proxy for ring membership — no community column exists. The result is a popularity ranking of merchants among heavy spenders. It sounds like a real answer.

Against the enriched catalog, the question becomes: which merchants are most commonly visited by accounts in ring-candidate communities? That answer is a different list — not popular merchants, but the merchants where ring members cluster disproportionately.

In [ ]:
ANCHOR1_Q = (
    "Which merchants are most commonly visited by accounts with the highest "
    "total transaction volume?"
)

print("[1] merchant_favorites — asking...")
r1 = ask_genie(ANCHOR1_Q)

if r1["df"] is not None:
    print(f"    Rows: {len(r1['df'])}")
    print(f"    SQL:  {r1['sql']}")
    display(r1["df"])
else:
    print(f"    Status: {r1['status']}")
    print(f"    Text:   {r1['text']}")

### 2. Book Share

How much total balance do the top 10% of accounts by transfer volume hold, and what share of the book is that? Volume decile is the best available proxy for ring membership on the base catalog.

Against the enriched catalog: total balance held by ring-candidate community members, and their share of the book. That is the number risk leadership asks for.

In [ ]:
ANCHOR2_Q = (
    "For the top 10% of accounts by transfer volume, what is the total balance "
    "held and what share of the book do they represent?"
)

print("[2] book_share_by_volume — asking...")
r2 = ask_genie(ANCHOR2_Q)

if r2["df"] is not None:
    print(f"    Rows: {len(r2['df'])}")
    print(f"    SQL:  {r2['sql']}")
    display(r2["df"])
else:
    print(f"    Status: {r2['status']}")
    print(f"    Text:   {r2['text']}")

### 3. Investigator Review Queue

How many accounts are in the top 10% by transfer volume, and how do they break down by region? Without a risk tier column, a volume cutoff is the best available threshold for sizing an investigation workload.

Against the enriched catalog: how many accounts are in the high risk tier, and what is the regional breakdown? A risk manager immediately sees the difference — volume cutoff versus a defensible structural segment.

In [ ]:
ANCHOR3_Q = (
    "How many accounts are in the top 10% by transfer volume, "
    "and what is the regional breakdown?"
)

print("[3] review_queue_by_volume — asking...")
r3 = ask_genie(ANCHOR3_Q)

if r3["df"] is not None:
    print(f"    Rows: {len(r3['df'])}")
    print(f"    SQL:  {r3['sql']}")
    display(r3["df"])
else:
    print(f"    Status: {r3['status']}")
    print(f"    Text:   {r3['text']}")

### 4. Internal vs External Transfer Ratio

What fraction of transfer volume flows between accounts that have transacted with each other more than five times, versus accounts with no prior relationship? Repeat-transfer frequency is a proxy for community insularity — money that keeps circulating within a closed group.

Against the enriched catalog: per-community internal versus external ratios, measured against actual community membership. The proxy and the structural answer are asking the same thing; the numbers will differ.

In [ ]:
ANCHOR4_Q = (
    "What fraction of total transfer volume flows between accounts that have transacted "
    "with each other more than five times, versus accounts with no prior relationship?"
)

print("[4] internal_vs_external_ratio — asking...")
r4 = ask_genie(ANCHOR4_Q)

if r4["df"] is not None:
    print(f"    Rows: {len(r4['df'])}")
    print(f"    SQL:  {r4['sql']}")
    display(r4["df"])
else:
    print(f"    Status: {r4['status']}")
    print(f"    Text:   {r4['text']}")

### 5. Merchant Community Concentration

Are there merchants where the majority of transaction volume comes from accounts that also transact heavily with each other? Co-transaction frequency approximates community cohesion using only the base tables.

Against the enriched catalog: merchants whose customer base is disproportionately concentrated in a single community. Co-transaction frequency and actual community membership are measuring similar things by different means; which merchants appear, and how concentrated they look, will differ.

In [ ]:
ANCHOR5_Q = (
    "Are there merchants where the majority of transaction volume comes from "
    "accounts that also transact heavily with each other?"
)

print("[5] merchant_concentration_by_coactivity — asking...")
r5 = ask_genie(ANCHOR5_Q)

if r5["df"] is not None:
    print(f"    Rows: {len(r5['df'])}")
    print(f"    SQL:  {r5['sql']}")
    display(r5["df"])
else:
    print(f"    Status: {r5['status']}")
    print(f"    Text:   {r5['text']}")

In [ ]:
def _status(r):
    if r["df"] is not None and len(r["df"]) > 0:
        return "RESPONDED"
    elif r["df"] is not None:
        return "NO DATA"
    return "NO DATA" if r["text"] else "ERROR"

results = [
    ("merchant_favorites",              r1),
    ("book_share_by_volume",            r2),
    ("review_queue_by_volume",          r3),
    ("internal_vs_external_ratio",      r4),
    ("merchant_concentration_by_coactivity", r5),
]

print("=" * 78)
print("Genie BEFORE — Summary")
print("=" * 78)
print()
print("These are the best answers the base catalog can produce.")
print("Each uses a volume or frequency proxy for the structural dimension it cannot reach.")
print()
for name, r in results:
    print(f"  {_status(r):<12}  {name}")
print()
print("-" * 78)
print("Run the pipeline (02 → 03 → 04), then open Genie against the enriched")
print("Gold tables and ask the same questions. The anchor gap will be visible.")

## Next Steps

Proceed to **`02_neo4j_ingest`** to load the five base tables into Neo4j as a property graph. After the full pipeline (`02_neo4j_ingest` → `03_gds_enrichment` → `04_pull_gold_tables`), open Genie against the enriched Gold tables and ask the same five questions. Use `genie-guide.md` for the copy-paste versions of the after questions.